# 10.1 - Function Calling for MCP

**Module 10 - Tool Use & MCP** | Netsetos GenAI Engineering

This notebook builds the function-calling primitive from the ground up: the four-step tool-use loop, the three provider shapes that all implement it, and the reliability and safety layers around it. Every cell is a tiny self-contained simulation - no live API keys required - so you can run the whole thing and watch how a model requests a tool while your code does the executing.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This first cell just prepares the environment. Everything in this notebook is a pure-Python simulation of the tool-use loop, so nothing here reaches out to a real model - the `pip install` line is commented out and only needed if you later run the illustrative live-API snippets.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

Environment prep only. The two SDKs (`anthropic`, `openai`) are named for the reference live-API code shown later in the lesson, but none of the runnable cells import them - the whole notebook runs on the standard library.

**How the code works - step by step**
- **`# !pip install anthropic openai -q`** - commented out; uncomment only if you plan to run the illustrative provider snippets against real keys.
- **reproducibility comment** - a note that the lesson uses seeded/fixed values; the runnable cells are deterministic simulations, so no randomness is actually configured here.

**In one line:** nothing executes - it is a placeholder for optional dependencies.

**What you'll see:** (no output - environment prep)

## 1 - The tool-use loop

Strip away every provider difference and tool calling is always the same four steps: you send the message plus tool schemas, the model REQUESTS a call (name + args), YOUR code runs the real function, and the result goes back for the final answer. This cell simulates all four with a keyword-router standing in for the model, so the loop shape is unmistakable.

In [ ]:
# THE TOOL-USE LOOP - the model REQUESTS a call; YOUR code runs it and feeds the result back.
import ast, operator
def eval_safe(expr):                              # tiny safe arithmetic (no eval, no builtins)
    OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
    def ev(n):
        if isinstance(n, ast.Constant): return n.value
        if isinstance(n, ast.BinOp):    return OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(ast.parse(expr, mode="eval").body)

def mock_model_pick(msg):                          # stands in for the LLM (keyword router)
    m = msg.lower()
    if "weather" in m: return {"tool": "get_weather", "args": {"city": "Hyderabad"}}
    if any(c.isdigit() for c in m): return {"tool": "calculate", "args": {"expr": "14999*1.18"}}
    return {"tool": None, "answer": "I can answer that directly."}

def run_tool(name, args):
    if name == "get_weather": return {"temp_c": 34, "cond": "Sunny"}
    if name == "calculate":   return {"result": round(eval_safe(args["expr"]), 2)}

def tool_use_loop(msg):
    print(f"  1. user -> model:   {msg!r}")
    step = mock_model_pick(msg)
    if not step["tool"]:
        print(f"  2. model -> answer: {step['answer']} (no tool needed)"); return
    print(f"  2. model REQUESTS:  {step['tool']}({step['args']})   <- a request, not an execution")
    print(f"  3. YOUR CODE runs it -> {run_tool(step['tool'], step['args'])}")
    print(f"  4. result -> model -> final natural-language answer")

tool_use_loop("what's the weather in Hyderabad?")
tool_use_loop("what is 14999 plus 18% GST?")

# Output:
#   1. user -> model:   "what's the weather in Hyderabad?"
#   2. model REQUESTS:  get_weather({'city': 'Hyderabad'})   <- a request, not an execution
#   3. YOUR CODE runs it -> {'temp_c': 34, 'cond': 'Sunny'}
#   4. result -> model -> final natural-language answer
#   1. user -> model:   'what is 14999 plus 18% GST?'
#   2. model REQUESTS:  calculate({'expr': '14999*1.18'})   <- a request, not an execution
#   3. YOUR CODE runs it -> {'result': 17698.82}
#   4. result -> model -> final natural-language answer

**Explanation**

A skeleton of the whole lesson, not a model call. A fake `mock_model_pick` decides which tool a message needs, `run_tool` executes it, and `tool_use_loop` prints the four hops. The point is the split: the model is a planner that emits a request; your code is the executor that runs it and feeds the result back.

**How the code works - step by step**
- **`eval_safe`** - an AST-based arithmetic evaluator (only `+ - * /` on numbers), previewing Cell 6's rule: never `eval` model-supplied strings.
- **`mock_model_pick`** - the LLM stand-in; a keyword router that returns a tool name + args (weather or calculate) or a direct answer.
- **`run_tool`** - YOUR code; returns a hard-coded weather reading or the safe-evaluated calculation.
- **`tool_use_loop`** - prints the four hops: user→model, model REQUESTS (not executes), your code runs it, result→model→final answer.

**In one line:** model requests -> your code runs -> result goes back.

**What you'll see:** Two traced runs. The weather message prints a `get_weather({'city': 'Hyderabad'})` request then `{'temp_c': 34, 'cond': 'Sunny'}`; the GST message prints a `calculate({'expr': '14999*1.18'})` request then `{'result': 17698.82}` - each labelled as a request, not an execution.

## 2 - The same tool, three provider shapes

The concept is identical across vendors; only the plumbing differs. This cell builds one weather tool in the Anthropic and OpenAI shapes side by side and notes how google-genai skips schemas entirely by taking the Python function itself.

In [ ]:
# THE SAME TOOL, THREE PROVIDER SHAPES (the concept is identical; the field names differ).
def anthropic_tool(name, desc, schema):
    return {"name": name, "description": desc, "input_schema": schema}          # -> content[].type=="tool_use"
def openai_tool(name, desc, schema):
    return {"type": "function", "function": {"name": name, "description": desc, "parameters": schema}}  # -> message.tool_calls[]
# google-genai: you pass the PYTHON FUNCTION itself; the SDK builds the schema from hints+docstring.

schema = {"type": "object",
          "properties": {"city": {"type": "string", "description": "City name"},
                         "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}},
          "required": ["city"]}
a = anthropic_tool("get_weather", "Get current weather for a city", schema)
o = openai_tool("get_weather", "Get current weather for a city", schema)
print("  Anthropic: schema key =", list(a.keys())[-1], "| response = content[].type == 'tool_use'")
print("  OpenAI:    schema key =", list(o['function'].keys())[-1], "| response = message.tool_calls[] (.arguments is a JSON string)")
print("  Google:    pass the python function; automatic function calling is ON by default")

# Output:
#   Anthropic: schema key = input_schema | response = content[].type == 'tool_use'
#   OpenAI:    schema key = parameters | response = message.tool_calls[] (.arguments is a JSON string)
#   Google:    pass the python function; automatic function calling is ON by default

**Explanation**

A shape-comparison harness, not a model call. Two builder functions wrap the exact same JSON schema in each provider's envelope, then the cell prints where each provider hides the schema and how each returns its tool request - the mismatch that MCP later standardizes away.

**How the code works - step by step**
- **`anthropic_tool`** - nests the schema under `input_schema`; responses come back as `content[]` blocks with `type=='tool_use'`.
- **`openai_tool`** - wraps it as `{type:'function', function:{parameters:...}}`; responses come back in `message.tool_calls[]`, with `.arguments` as a JSON *string* you must `json.loads`.
- **`schema`** - one shared JSON schema (`city` required, `unit` an enum) passed to both builders to prove the body is identical.
- **google-genai note** - no builder; you pass the Python function and the SDK infers the schema from type hints + docstring.

**In one line:** same schema, three envelopes - `input_schema` vs `parameters` vs the function itself.

**What you'll see:** Three lines confirming the schema key per provider: Anthropic `input_schema` (response `content[].type=='tool_use'`), OpenAI `parameters` (response `message.tool_calls[]`, arguments a JSON string), and Google (pass the function; automatic function calling on by default).

## 3 - Schema design is the reliability lever

Tool-calling accuracy is mostly a schema problem, not a model problem. The model reads your **description** to decide WHEN to call and your **schema** to fill the args. This cell scores two schemas on four concrete levers so you can see the gap.

In [ ]:
# SCHEMA QUALITY -> TOOL-CALL RELIABILITY. The model reads descriptions to decide WHEN to call.
def schema_score(tool):
    pts, notes = 0, []
    d = tool.get("description", "")
    if len(d) >= 20 and d.lower() not in ("weather tool", "tool"): pts += 30; notes.append("good description (+30)")
    else: notes.append("weak description (0) - the model reads this to decide WHEN to call")
    props = tool.get("input_schema", {}).get("properties", {})
    if any("enum" in p for p in props.values()): pts += 25; notes.append("enum constrains values (+25)")
    else: notes.append("no enum -> model may hallucinate values")
    if tool.get("input_schema", {}).get("required"): pts += 25; notes.append("required fields marked (+25)")
    else: notes.append("nothing required -> model may skip critical args")
    if "_" in tool["name"] and tool["name"].split("_")[0] in ("get","create","search","update","delete","list","send"):
        pts += 20; notes.append("verb_noun name (+20)")
    else: notes.append("unclear name")
    return pts, notes

good = {"name":"get_weather","description":"Get the current weather for a specific city",
        "input_schema":{"properties":{"unit":{"enum":["celsius","fahrenheit"]}},"required":["city"]}}
bad  = {"name":"weather","description":"weather tool","input_schema":{"properties":{"c":{"type":"string"}}}}
for label, t in [("GOOD schema", good), ("BAD schema", bad)]:
    s, notes = schema_score(t)
    print(f"  {label}: reliability score {s}/100")
    for n in notes: print(f"      - {n}")

# Output:
#   GOOD schema: reliability score 100/100
#       - good description (+30)
#       - enum constrains values (+25)
#       - required fields marked (+25)
#       - verb_noun name (+20)
#   BAD schema: reliability score 0/100
#       - weak description (0) - the model reads this to decide WHEN to call
#       - no enum -> model may hallucinate values
#       - nothing required -> model may skip critical args
#       - unclear name

**Explanation**

A scoring rubric, not a model call. `schema_score` awards points for the four things that actually move tool-call reliability, then runs a deliberately good and a deliberately bad schema through it. The 0-vs-100 spread is a stand-in for the real accuracy difference a vague schema causes.

**How the code works - step by step**
- **description check** - +30 if the description is specific (>=20 chars and not a generic 'weather tool'/'tool'); this is what the model reads to decide WHEN to call.
- **enum check** - +25 if any property constrains values with an `enum`, so the model can't invent one.
- **required check** - +25 if `required` is set, so the model can't skip critical args.
- **name check** - +20 for a `verb_noun` name whose verb is a known action (get/create/search/...).
- **`good` vs `bad`** - a fully-specified schema and a bare `{'description':'weather tool'}` run through the scorer.

**In one line:** description + enum + required + verb_noun name = a reliable tool.

**What you'll see:** The GOOD schema scores 100/100 with all four bonuses itemized; the BAD schema scores 0/100 with each miss explained (weak description, no enum, nothing required, unclear name).

## 4 - tool_choice and parallel calls

Two knobs give you most of your behavioral control over tool use: `tool_choice` decides WHEN the model may call tools, and parallel calling decides HOW MANY it can request at once. This cell maps each mode to its rule and contrasts sequential vs parallel latency.

In [ ]:
# tool_choice MODES + parallel vs sequential latency.
def what_can_the_model_do(mode):
    return {"auto":"call a tool OR answer directly (default)",
            "required/any":"MUST call at least one tool this turn",
            "forced":"MUST call the ONE named tool (deterministic routing)",
            "none":"answer in text only; tools ignored"}[mode]
for m in ("auto","required/any","forced","none"):
    print(f"  tool_choice={m:14s} -> {what_can_the_model_do(m)}")

def latency(n, per_ms=200, parallel=False):
    return per_ms if parallel else per_ms * n                    # parallel = one round-trip
for n in (1, 3, 5):
    print(f"  {n} independent calls: sequential {latency(n)}ms vs parallel {latency(n, parallel=True)}ms")

# Output:
#   tool_choice=auto           -> call a tool OR answer directly (default)
#   tool_choice=required/any   -> MUST call at least one tool this turn
#   tool_choice=forced         -> MUST call the ONE named tool (deterministic routing)
#   tool_choice=none           -> answer in text only; tools ignored
#   1 independent calls: sequential 200ms vs parallel 200ms
#   3 independent calls: sequential 600ms vs parallel 200ms
#   5 independent calls: sequential 1000ms vs parallel 200ms

**Explanation**

A behavior + latency lookup, not a model call. One function translates each `tool_choice` mode into plain English; another computes round-trip latency, showing that independent calls collapse to a single round-trip when run in parallel.

**How the code works - step by step**
- **`what_can_the_model_do`** - maps each mode to its rule: `auto` (call or answer), `required`/`any` (must call), `forced` (one named tool), `none` (text only).
- **mode loop** - prints all four modes with their behavior.
- **`latency`** - returns `per_ms` once when `parallel=True`, else `per_ms * n` - parallel is one round-trip regardless of count.
- **count loop** - compares sequential vs parallel totals for 1, 3, and 5 independent calls.

**In one line:** `tool_choice` controls WHEN; parallel calling controls HOW MANY at once.

**What you'll see:** A table of the four tool_choice modes with their rules, then latency rows: 3 calls drop from 600ms sequential to 200ms parallel, and 5 calls from 1000ms to 200ms (one call is 200ms either way).

## 5 - A universal, safe ToolExecutor

Since the loop is identical across providers, wrap it once. This cell builds a tiny `ToolExecutor` that registers Python functions and runs them by name - and crucially returns errors instead of swallowing them, using an AST-based safe evaluator instead of `eval`.

In [ ]:
# A UNIVERSAL ToolExecutor - register once, run with any provider. SAFE evaluator (no eval).
import ast, operator
def eval_safe(expr):
    OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
    def ev(n):
        if isinstance(n, ast.Constant): return n.value
        if isinstance(n, ast.BinOp):    return OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(ast.parse(expr, mode="eval").body)

class ToolExecutor:
    def __init__(self): self.tools = {}
    def register(self, func): self.tools[func.__name__] = func; return func
    def execute(self, name, args):
        f = self.tools.get(name)
        if not f: return {"error": f"unknown tool: {name}"}
        try: return {"ok": f(**args)}
        except Exception as e: return {"error": str(e)}    # errors go BACK to the model, never swallowed

ex = ToolExecutor()
@ex.register
def get_weather(city):
    return {"Hyderabad": {"temp_c": 34}, "Mumbai": {"temp_c": 30}}.get(city, {"error": "unknown city"})
@ex.register
def calculate(expr):
    return {"result": round(eval_safe(expr), 2)}

print("    get_weather('Hyderabad') ->", ex.execute("get_weather", {"city": "Hyderabad"}))
print("    calculate('14999*1.18')  ->", ex.execute("calculate", {"expr": "14999*1.18"}))
print("    unknown tool             ->", ex.execute("nope", {}))

# Output:
#     get_weather('Hyderabad') -> {'ok': {'temp_c': 34}}
#     calculate('14999*1.18')  -> {'ok': {'result': 17698.82}}
#     unknown tool             -> {'error': 'unknown tool: nope'}

**Explanation**

A small reusable registry, and the pattern an MCP server later wraps. `register` decorates functions into a name->function dict; `execute` looks one up and calls it, wrapping the result in `{'ok':...}` or `{'error':...}` so failures flow back to the model instead of vanishing.

**How the code works - step by step**
- **`eval_safe`** - the same AST arithmetic evaluator; a model-supplied `__import__(...)` raises rather than running.
- **`ToolExecutor.__init__`** - creates an empty `tools` dict.
- **`register`** - a decorator that stores a function by its `__name__` and returns it.
- **`execute`** - looks up the name; returns `{'error': 'unknown tool'}` if missing, else calls `f(**args)` inside try/except so exceptions come back as `{'error': str(e)}` - never swallowed.
- **`get_weather` / `calculate`** - two registered tools; weather is a lookup, calculate routes through `eval_safe`.

**In one line:** register once, execute by name, and always return errors so the model can self-correct.

**What you'll see:** Three lines: `get_weather('Hyderabad')` -> `{'ok': {'temp_c': 34}}`, `calculate('14999*1.18')` -> `{'ok': {'result': 17698.82}}`, and an unknown tool -> `{'error': 'unknown tool: nope'}`.

## 6 - Don't trust the args - validate and self-correct

The single most common production tool-calling failure is silently defaulting bad arguments and running the tool anyway. This cell contrasts the fix - hand the validation error back to the model so it retries - against that anti-pattern.

In [ ]:
# BAD-JSON: self-correction (the fix) vs silent-fallback (the #1 anti-pattern).
def validate(args, required):
    missing = [k for k in required if k not in args]
    return (True, "") if not missing else (False, f"missing required field(s): {missing}")

def with_self_correction(model_attempts, required, max_tries=3):
    for i, args in enumerate(model_attempts[:max_tries], 1):
        ok, err = validate(args, required)
        if ok: return f"attempt {i}: valid {args} -> execute"
        print(f"    attempt {i}: {args} -> INVALID ({err}) -> return the error to the model, retry")
    return "gave up after retries -> graceful error to the user"

print("  THE FIX (return the error to the model; it self-corrects):")
print("   ", with_self_correction([{}, {"city": "Hyderabad"}], required=["city"]))
print("  THE ANTI-PATTERN: validate fails -> args={} -> tool runs with no city -> WRONG result, silently")

# Output:
#   THE FIX (return the error to the model; it self-corrects):
#     attempt 1: {} -> INVALID (missing required field(s): ['city']) -> return the error to the model, retry
#     attempt 2: valid {'city': 'Hyderabad'} -> execute
#   THE ANTI-PATTERN: validate fails -> args={} -> tool runs with no city -> WRONG result, silently

**Explanation**

A validation-loop demo, not a model call. `validate` checks required fields; `with_self_correction` walks a list of model 'attempts', returning the error to the model on each failure until one validates or the retry budget is spent. It shows the model fixing itself from the error message.

**How the code works - step by step**
- **`validate`** - returns `(False, message)` listing any missing required fields, else `(True, '')`.
- **`with_self_correction`** - iterates up to `max_tries` model attempts; on an invalid one it prints that the error is returned to the model and retries; on a valid one it executes; if all fail it degrades to a graceful user error.
- **the two prints** - run the fix path (`[{}, {'city':'Hyderabad'}]`), then describe the anti-pattern where a failed validation defaults to `{}` and runs the tool with no city.

**In one line:** invalid args -> return the error to the model -> it retries -> execute; never silently default.

**What you'll see:** The fix path prints attempt 1 (`{}`) as INVALID (missing 'city') then attempt 2 (`{'city': 'Hyderabad'}`) as valid -> execute, followed by a one-line description of the silent-fallback anti-pattern.

## 7 - From function calling to MCP

Function calling is powerful but vendor-specific: N apps each writing a connector to M tools is N×M bespoke integrations. MCP standardizes the primitive over JSON-RPC 2.0 so each tool is one server any client connects to - turning N×M into N+M. This cell quantifies the saving.

In [ ]:
# FUNCTION CALLING -> MCP: the N x M integration problem, and how MCP makes it N + M.
def integrations(n_clients, m_tools, mcp):
    return (n_clients + m_tools) if mcp else (n_clients * m_tools)

print("  Without MCP each app writes its own connector to each tool: N x M bespoke integrations.")
print("  With MCP each tool is ONE server speaking JSON-RPC 2.0; any client connects: N + M.")
print(f"  {'clients':>8} {'tools':>6} {'no MCP (NxM)':>13} {'MCP (N+M)':>10} {'saved':>7}")
for n, m in [(3, 4), (5, 10), (10, 50)]:
    w, mc = integrations(n, m, False), integrations(n, m, True)
    print(f"  {n:>8} {m:>6} {w:>13} {mc:>10} {w-mc:>7}")
print("  MCP primitives: Tools (invoke), Resources (read), Prompts (templates). Under the hood: still function calls.")

# Output:
#   Without MCP each app writes its own connector to each tool: N x M bespoke integrations.
#   With MCP each tool is ONE server speaking JSON-RPC 2.0; any client connects: N + M.
#    clients  tools  no MCP (NxM)  MCP (N+M)   saved
#          3      4            12          7       5
#          5     10            50         15      35
#         10     50           500         60     440
#   MCP primitives: Tools (invoke), Resources (read), Prompts (templates). Under the hood: still function calls.

**Explanation**

An arithmetic argument for MCP, not a model call. `integrations` computes `n*m` without MCP versus `n+m` with it, then tabulates the gap across three scales. The takeaway: an MCP tool call is still a function call underneath - MCP just standardizes the wiring.

**How the code works - step by step**
- **`integrations`** - returns `n_clients + m_tools` when `mcp=True`, else `n_clients * m_tools`.
- **explanatory prints** - state the N×M bespoke problem and the N+M MCP fix (each tool one JSON-RPC 2.0 server, any client connects).
- **the table loop** - runs (3,4), (5,10), (10,50), printing no-MCP vs MCP counts and the integrations saved.
- **primitives line** - names MCP's three primitives (Tools/Resources/Prompts) and notes it is still function calls underneath.

**In one line:** N×M bespoke connectors collapse to N+M once every tool speaks one standard (JSON-RPC 2.0).

**What you'll see:** A table showing the saving grow with scale: 3×4 goes 12 -> 7 (saved 5), 5×10 goes 50 -> 15 (saved 35), 10×50 goes 500 -> 60 (saved 440), plus a line listing the MCP primitives.

## 8 - India function calling: translate-first

For Indic apps the stack is concrete: Sarvam (OpenAI-compatible tool use + an efficient Indic tokenizer), Bhashini (22-language speech and translation as tools), and the translate-first pattern - route tools in English for reliability, keep the user experience fully Indic. This cell lays out the pipeline and the token saving.

In [ ]:
# INDIA: translate-first Hindi voice agent (each piece plays to its strength).
def indic_pipeline(lang):
    return [f"Bhashini STT ({lang} audio -> text)", f"detect language = {lang}",
            f"Sarvam translate ({lang} -> English)", "function-call in ENGLISH (tool routing far more accurate)",
            "run tools", f"translate results (English -> {lang})", f"Bhashini TTS (text -> {lang} audio)"]
def gpt_vs_sarvam_tokens(hindi_chars):
    return {"gpt_style": hindi_chars // 2, "sarvam": hindi_chars // 4}   # Sarvam Indic tokenizer ~half (ballpark)

for i, s in enumerate(indic_pipeline("Hindi"), 1): print(f"    {i}. {s}")
t = gpt_vs_sarvam_tokens(400)
print(f"  ~400 Hindi chars: GPT-style ~{t['gpt_style']} tokens vs Sarvam ~{t['sarvam']} tokens (~half).")

# Output:
#     1. Bhashini STT (Hindi audio -> text)
#     2. detect language = Hindi
#     3. Sarvam translate (Hindi -> English)
#     4. function-call in ENGLISH (tool routing far more accurate)
#     5. run tools
#     6. translate results (English -> Hindi)
#     7. Bhashini TTS (text -> Hindi audio)
#   ~400 Hindi chars: GPT-style ~200 tokens vs Sarvam ~100 tokens (~half).

**Explanation**

A pipeline blueprint plus a token estimate, not a model call. `indic_pipeline` returns the ordered stages of a Hindi voice agent; `gpt_vs_sarvam_tokens` gives a ballpark showing Sarvam's Indic tokenizer using roughly half the tokens of GPT-style models on Hindi.

**How the code works - step by step**
- **`indic_pipeline`** - returns the 7 stages: Bhashini STT -> detect language -> Sarvam translate to English -> function-call in English (routing is far more accurate there) -> run tools -> translate results back -> Bhashini TTS.
- **`gpt_vs_sarvam_tokens`** - a rough model: `chars//2` for GPT-style, `chars//4` for Sarvam, i.e. about half the tokens on Hindi.
- **the two loops** - print the numbered pipeline, then the token comparison for ~400 Hindi characters.

**In one line:** translate to English to route tools reliably, then translate back - each layer plays to its strength.

**What you'll see:** The 7-step Hindi pipeline printed in order, then a token line: ~400 Hindi chars are roughly 200 tokens GPT-style vs ~100 with Sarvam (about half).

Across nine cells you built the entire function-calling picture without a single live API call: the model REQUESTS, your code RUNS, the result flows back (Cell 1), and every provider is just a different set of field names around that one loop (Cell 2). A good schema is the cheapest reliability win (Cell 3), `tool_choice` and parallel calls are your control knobs (Cell 4), a universal safe executor makes it portable (Cell 5), and validate-then-self-correct plus least-privilege is the non-negotiable safety layer (Cell 6). Standardize that primitive over JSON-RPC and N×M integrations collapse to N+M - that is MCP (Cell 7) - and the same loop runs translate-first in 22 Indic languages (Cell 8). Next, Lesson 10.2 turns this primitive into a real MCP server and client.